<a href="https://colab.research.google.com/github/CodeHunterOfficial/ArabovMKDeep/blob/main/NLP-2026/Lecture_2/ELMo_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ELMo: контекстуализированные эмбеддинги на основе языковых моделей

## Введение: почему ELMo стал поворотным моментом

Мы прошли долгий путь от one-hot encoding до FastText и Doc2Vec. Все эти методы объединяет одна фундаментальная особенность: каждое слово имеет **один фиксированный вектор**, независимо от контекста, в котором оно встречается. Слово «банк» в предложениях «Я пошёл в банк за деньгами» и «Я сидел на берегу реки и смотрел на банк» получает **одинаковый** вектор, хотя означает совершенно разные вещи. Это называется проблемой **полисемии** — многозначности слов.

Word2Vec, GloVe, FastText — все они дают статические эмбеддинги. Они не различают значения слова в разных контекстах. Это фундаментальное ограничение, которое долгое время считалось неизбежным компромиссом.

**ELMo** (Embeddings from Language Models), предложенный Мэтью Петерсом и его коллегами из Allen Institute for AI и University of Washington в 2018 году, изменил это. ELMo стал одним из первых методов, который генерирует **контекстуализированные** эмбеддинги: вектор слова зависит от всего предложения, в котором оно встречается. Одно и то же слово в разных контекстах получает **разные** векторы.

Это не просто техническое улучшение. Это смена парадигмы. ELMo показал, что можно предобучить глубокую языковую модель на большом корпусе, а затем использовать её внутренние представления для широкого спектра задач NLP, достигая state-of-the-art результатов. Это заложило основу для BERT и всей эпохи предобученных языковых моделей.

В этой лекции мы подробно разберём:

- что такое языковое моделирование и как оно связано с эмбеддингами;
- архитектуру ELMo: символьные свёртки, двунаправленный LSTM;
- прямое и обратное языковое моделирование;
- как ELMo комбинирует слои для получения контекстуализированных эмбеддингов;
- как обучать ELMo;
- как использовать ELMo в downstream-задачах;
- в чём отличие ELMo от статических эмбеддингов;
- ограничения и расширения ELMo.

---

## 1. Языковое моделирование как основа ELMo

### 1.1 Что такое языковая модель?

**Языковая модель** (language model, LM) — это вероятностная модель, которая оценивает вероятность последовательности слов:

$$
P(w_1, w_2, \ldots, w_T),
$$

где $w_1, \ldots, w_T$ — последовательность слов. По цепному правилу вероятности:

$$
P(w_1, \ldots, w_T) = \prod_{t=1}^{T} P(w_t \mid w_1, \ldots, w_{t-1}).
$$

Это означает, что вероятность всей последовательности раскладывается в произведение вероятностей каждого слова при условии всех предыдущих слов.

**Задача языкового моделирования:** по предыдущим словам предсказать следующее слово. Например, по словам «Я пошёл в» предсказать «магазин» или «парк» с некоторой вероятностью.

### 1.2 Прямое и обратное языковое моделирование

**Прямая языковая модель** (forward LM) предсказывает следующее слово по предыдущим:

$$
P(w_1, \ldots, w_T) = \prod_{t=1}^{T} P(w_t \mid w_1, \ldots, w_{t-1}).
$$

**Обратная языковая модель** (backward LM) предсказывает предыдущее слово по последующим:

$$
P(w_1, \ldots, w_T) = \prod_{t=1}^{T} P(w_t \mid w_{t+1}, \ldots, w_T).
$$

**Тонкий момент:** обратная языковая модель не «читает текст справа налево» в смысле понимания. Она просто использует будущий контекст для предсказания предыдущего слова. В комбинации с прямой моделью она даёт **двунаправленный** контекст.

### 1.3 Связь языкового моделирования и эмбеддингов

Почему языковое моделирование полезно для эмбеддингов? Ответ прост: чтобы хорошо предсказывать следующее слово, модель должна **понимать** язык. Она должна улавливать синтаксис (порядок слов, согласование), семантику (значения слов, тематику) и прагматику (контекст, намерение). Внутренние представления модели, которые она использует для предсказания, неизбежно кодируют эту информацию.

**Ключевая идея ELMo:** мы обучаем глубокую двунаправленную языковую модель на большом корпусе, а затем используем её **внутренние скрытые состояния** как эмбеддинги слов. Поскольку модель обучена предсказывать слова в контексте, её скрытые состояния содержат контекстуализированную информацию о каждом слове.

---

## 2. Архитектура ELMo

### 2.1 Общая схема

ELMo состоит из трёх основных компонентов:

1. **Символьный слой (Character CNN):** преобразует каждый токен (слово) в контекстно-независимый вектор.
2. **Двунаправленный LSTM (biLSTM):** обрабатывает последовательность векторов слов в прямом и обратном направлениях.
3. **Комбинирующий слой:** объединяет скрытые состояния всех слоёв biLSTM в единый контекстуализированный эмбеддинг.

**Схема:**

```
Входное предложение: "The cat sat on the mat"
         ↓
[Character CNN] → контекстно-независимые векторы слов x_1, x_2, ..., x_T
         ↓
[Forward LSTM]  → h_1^→, h_2^→, ..., h_T^→
[Backward LSTM] → h_1^←, h_2^←, ..., h_T^←
         ↓
[Комбинирование] → контекстуализированные эмбеддинги ELMo_1, ..., ELMo_T
```

### 2.2 Символьный слой (Character CNN)

**Проблема:** словарь может содержать миллионы слов. Если использовать one-hot или обычный embedding layer, мы столкнёмся с проблемой OOV (out-of-vocabulary) и огромным числом параметров.

**Решение ELMo:** использовать **символьные свёртки** для построения вектора слова из его символов.

**Как это работает:**

1. Каждый символ слова преобразуется в вектор размерности 16 (character embedding).
2. Применяются одномерные свёртки с 2048 фильтрами ширины от 1 до 7 символов.
3. Результаты свёрток подвергаются max-pooling по позициям символов.
4. Полученный вектор проходит через два highway-слоя.
5. Линейная проекция до размерности 512.

**Формально:** для слова $w$ с символами $c_1, c_2, \ldots, c_{|w|}$:

$$
x_w = \text{Highway}(\text{MaxPool}(\text{Conv}([e_{c_1}; e_{c_2}; \ldots; e_{c_{|w|}}]))).
$$

где $e_{c_i} \in \mathbb{R}^{16}$ — вектор символа $c_i$.

**Преимущества:**

- **Нет OOV:** даже если слово не встречалось в корпусе, его символы, скорее всего, встречались.
- **Морфология:** слова с общими морфемами (приставки, суффиксы) получают похожие векторы.
- **Опечатки:** слово «кошкаа» (с опечаткой) имеет символы, общие с «кошкой», поэтому вектор будет близок.

**Тонкий момент:** в отличие от FastText, где n-граммы хешируются и суммируются, ELMo использует свёртки, что позволяет улавливать более сложные паттерны символов.

### 2.3 Двунаправленный LSTM

**LSTM (Long Short-Term Memory)** — это рекуррентная нейронная сеть, способная улавливать долгосрочные зависимости в последовательностях. LSTM имеет механизмы «ворот» (gates), которые управляют потоком информации.

**Формально:** LSTM на каждом шаге $t$ получает вход $x_t$ и предыдущее скрытое состояние $h_{t-1}$, и вычисляет новое скрытое состояние $h_t$:

$$
\begin{aligned}
f_t &= \sigma(W_f [h_{t-1}; x_t] + b_f), \\
i_t &= \sigma(W_i [h_{t-1}; x_t] + b_i), \\
\tilde{C}_t &= \tanh(W_C [h_{t-1}; x_t] + b_C), \\
C_t &= f_t \odot C_{t-1} + i_t \odot \tilde{C}_t, \\
o_t &= \sigma(W_o [h_{t-1}; x_t] + b_o), \\
h_t &= o_t \odot \tanh(C_t).
\end{aligned}
$$

где $\sigma$ — сигмоида, $\odot$ — поэлементное умножение.

**Двунаправленный LSTM:** ELMo использует **два** LSTM:

- **Прямой LSTM:** обрабатывает последовательность слева направо: $h_1^\rightarrow, h_2^\rightarrow, \ldots, h_T^\rightarrow$.
- **Обратный LSTM:** обрабатывает последовательность справа налево: $h_1^\leftarrow, h_2^\leftarrow, \ldots, h_T^\leftarrow$.

**Ключевое отличие от Word2Vec:** в Word2Vec контекст — это фиксированное окно $m$. В ELMo контекст — это **всё предложение**. Прямой LSTM видит все предыдущие слова, обратный — все последующие. Это позволяет улавливать дальние зависимости.

**Тонкий момент:** ELMo использует **два слоя** LSTM в каждом направлении (всего 4 LSTM: 2 прямых, 2 обратных). Первый слой улавливает локальные зависимости, второй — более глобальные. Это ключевое отличие от однослойных моделей.

### 2.4 Прямое языковое моделирование

Прямая языковая модель предсказывает следующее слово по предыдущим:

$$
P(t_1, t_2, \ldots, t_N) = \prod_{k=1}^{N} P(t_k \mid t_1, \ldots, t_{k-1}).
$$

**Реализация:** последовательность векторов слов $x_1, x_2, \ldots, x_N$ подаётся в прямой LSTM. На каждом шаге $k$ LSTM выдаёт скрытое состояние $h_k^\rightarrow$. Это состояние используется для предсказания следующего слова $t_{k+1}$ через softmax:

$$
P(t_{k+1} \mid t_1, \ldots, t_k) = \text{softmax}(W_y h_k^\rightarrow + b_y).
$$

**Функция потерь для прямого LM:**

$$
\mathcal{L}_{\text{forward}} = -\sum_{k=1}^{N} \log P(t_{k+1} \mid t_1, \ldots, t_k).
$$

### 2.5 Обратное языковое моделирование

Обратная языковая модель предсказывает предыдущее слово по последующим:

$$
P(t_1, t_2, \ldots, t_N) = \prod_{k=1}^{N} P(t_k \mid t_{k+1}, \ldots, t_N).
$$

**Реализация:** та же последовательность $x_1, \ldots, x_N$ подаётся в обратный LSTM (который обрабатывает её справа налево). На каждом шаге $k$ обратный LSTM выдаёт скрытое состояние $h_k^\leftarrow$. Это состояние используется для предсказания предыдущего слова $t_{k-1}$:

$$
P(t_{k-1} \mid t_k, \ldots, t_N) = \text{softmax}(W_y h_k^\leftarrow + b_y).
$$

**Функция потерь для обратного LM:**

$$
\mathcal{L}_{\text{backward}} = -\sum_{k=1}^{N} \log P(t_k \mid t_{k+1}, \ldots, t_N).
$$

**Тонкий момент:** в оригинальной статье ELMo **разделяет** параметры для прямого и обратного LSTM, но **связывает** параметры для символьного слоя и softmax-слоя. Это означает, что оба направления используют одни и те же символьные эмбеддинги, но разные LSTM.

### 2.6 Общая функция потерь

Общая функция потерь для двунаправленной языковой модели:

$$
\mathcal{L}_{\text{biLM}} = \mathcal{L}_{\text{forward}} + \mathcal{L}_{\text{backward}}.
$$

Минимизация этой функции потерь заставляет модель учиться предсказывать слова как в прямом, так и в обратном направлении. В процессе обучения скрытые состояния LSTM становятся всё более информативными, кодируя синтаксис и семантику языка.

---

## 3. Контекстуализированные эмбеддинги ELMo

### 3.1 Скрытые состояния biLM

После предобучения biLM мы имеем для каждого токена $t_k$ набор скрытых состояний:

- $x_k^{LM}$ — контекстно-независимый вектор из символьного слоя;
- $\overrightarrow{h}_{k,j}^{LM}$ — прямое скрытое состояние слоя $j$ ($j = 1, \ldots, L$);
- $\overleftarrow{h}_{k,j}^{LM}$ — обратное скрытое состояние слоя $j$.

**Обозначение:** объединённое скрытое состояние на слое $j$:

$$
h_{k,j}^{LM} = [\overrightarrow{h}_{k,j}^{LM}; \overleftarrow{h}_{k,j}^{LM}], \quad j = 1, \ldots, L.
$$

и $h_{k,0}^{LM} = x_k^{LM}$ — контекстно-независимый вектор.

**Тонкий момент:** при $L = 2$ (два слоя LSTM в каждом направлении) мы имеем **три** уровня представлений:

- $h_{k,0}^{LM}$ — символьный слой;
- $h_{k,1}^{LM}$ — первый слой biLSTM;
- $h_{k,2}^{LM}$ — второй слой biLSTM.

Каждый из этих уровней содержит разную информацию. Первый слой улавливает синтаксис, второй — семантику.

### 3.2 Формула ELMo

Эмбеддинг ELMo для токена $t_k$ вычисляется как **взвешенная сумма** всех слоёв biLM:

$$
\text{ELMo}_k^{task} = \gamma^{task} \sum_{j=0}^{L} s_j^{task} \mathbf{h}_{k,j}^{LM},
$$

где:

- $s_j^{task}$ — веса, которые **обучаются** для каждой задачи отдельно;
- $\gamma^{task}$ — скалярный параметр, который масштабирует весь вектор ELMo;
- $\mathbf{h}_{k,j}^{LM}$ — скрытое состояние слоя $j$ для токена $k$.

**Разберём формулу по частям:**

1. **Сумма по слоям $j = 0, \ldots, L$:** мы объединяем информацию из всех слоёв biLM. Символьный слой ($j = 0$) даёт базовую морфологическую информацию, первый слой LSTM ($j = 1$) — синтаксис, второй слой ($j = 2$) — семантику.

2. **Веса $s_j^{task}$:** это **обучаемые** параметры. Они позволяют модели для каждой конкретной задачи решать, какие слои важнее. Например, для задачи POS-теггинга (части речи) важнее синтаксические слои, для задачи анализа тональности — семантические.

3. **Параметр $\gamma^{task}$:** это скаляр, который масштабирует вектор ELMo. Он нужен, потому что разные задачи могут требовать разной «громкости» ELMo-эмбеддингов.

**Тонкий момент:** веса $s_j^{task}$ нормализуются через softmax:

$$
s_j^{task} = \frac{\exp(w_j)}{\sum_{j'=0}^{L} \exp(w_{j'})}.
$$

Это гарантирует, что сумма весов равна 1, и их можно интерпретировать как важность каждого слоя.

**Интуиция:** ELMo не просто берёт последний слой biLM, как это делают многие модели. Он **комбинирует** все слои, позволяя downstream-модели самой решать, какая информация ей нужна. Это ключевое преимущество ELMo.

### 3.3 Свойства эмбеддингов ELMo

1. **Контекстуализированность:** вектор слова зависит от всего предложения. Слово «банк» в разных контекстах получает разные векторы.

2. **Многослойность:** ELMo использует информацию из всех слоёв biLM, а не только из последнего.

3. **Обучаемые веса:** веса $s_j^{task}$ и $\gamma^{task}$ обучаются для каждой задачи, что позволяет адаптировать ELMo к конкретным требованиям.

4. **Символьная основа:** ELMo не имеет проблемы OOV, потому что слова строятся из символов.

5. **Двунаправленность:** ELMo использует как левый, так и правый контекст, что даёт более полное понимание.

---

## 4. Обучение ELMo

### 4.1 Предобучение biLM

**Данные:** ELMo предобучается на большом корпусе текста. В оригинальной статье использовался корпус из примерно 30 миллионов предложений и 1 миллиарда слов. Это может быть Wikipedia, новостные статьи, книги и т.д.

**Архитектура:**

- Символьный слой: 2048 фильтров, ширина от 1 до 7, 2 highway-слоя, проекция до 512.
- LSTM: 2 слоя, 4096 единиц, с проекцией до 512 и residual connection между слоями.
- Обучение: Adam optimizer, learning rate 0.001.

**Функция потерь:** сумма прямого и обратного языкового моделирования:

$$
\mathcal{L}_{\text{biLM}} = -\sum_{k=1}^{N} \left[ \log P(t_{k+1} \mid t_1, \ldots, t_k) + \log P(t_{k-1} \mid t_k, \ldots, t_N) \right].
$$

**Тонкий момент:** при предобучении ELMo **разделяет** параметры для прямого и обратного LSTM, но **связывает** параметры для символьного слоя и softmax. Это означает, что оба направления используют одни и те же символьные эмбеддинги, но разные LSTM.

### 4.2 Заморозка параметров biLM

После предобучения параметры biLM **замораживаются**. Это означает, что веса символьного слоя и LSTM больше не обновляются. Они используются как **фиксированный экстрактор признаков**.

**Почему заморозка?** Предобучение на большом корпусе занимает много времени и ресурсов. Заморозка позволяет использовать предобученную модель для множества задач без переобучения. Это называется **transfer learning** (перенос обучения).

### 4.3 Обучение весов $s_j^{task}$ и $\gamma^{task}$

Для каждой downstream-задачи мы обучаем **только** веса $s_j^{task}$ и $\gamma^{task}$. Это небольшие параметры (всего $L+1$ весов плюс один скаляр), поэтому обучение быстрое и требует мало данных.

**Процесс:**

1. Заморозить biLM.
2. Для каждого токена в предложении вычислить все скрытые состояния $h_{k,j}^{LM}$.
3. Инициализировать $s_j^{task}$ равномерно (например, $1/(L+1)$) и $\gamma^{task} = 1$.
4. Обучить эти параметры на downstream-задаче вместе с остальной моделью.

**Тонкий момент:** веса $s_j^{task}$ могут быть **отрицательными**. Это означает, что некоторые слои могут «вычитаться» из общего представления. На практике это редкость, но теоретически возможно.

---

## 5. Использование ELMo в downstream-задачах

### 5.1 Общая схема

1. **Предобучить biLM** на большом корпусе.
2. **Заморозить** параметры biLM.
3. **Для каждой задачи:**
   - Пропустить входное предложение через biLM.
   - Собрать скрытые состояния $h_{k,j}^{LM}$ для всех токенов.
   - Вычислить ELMo-эмбеддинги как взвешенную сумму.
   - Подать ELMo-эмбеддинги в downstream-модель.
   - Обучить downstream-модель и веса $s_j^{task}, \gamma^{task}$.

### 5.2 Пример: классификация текстов

Для задачи классификации предложения:

1. Пропускаем предложение через biLM.
2. Получаем ELMo-эмбеддинги для каждого токена.
3. Усредняем ELMo-эмбеддинги по всем токенам:

$$
\text{ELMo}_{\text{sent}} = \frac{1}{N} \sum_{k=1}^{N} \text{ELMo}_k.
$$

4. Подаём $\text{ELMo}_{\text{sent}}$ в полносвязный слой с softmax для классификации.

**Тонкий момент:** ELMo можно использовать не только для эмбеддингов слов, но и для эмбеддингов предложений (усредняя или используя другие агрегации).

### 5.3 Пример: NER (Named Entity Recognition)

Для задачи NER:

1. Пропускаем предложение через biLM.
2. Получаем ELMo-эмбеддинги для каждого токена.
3. Подаём ELMo-эмбеддинги в BiLSTM-CRF модель.
4. Обучаем модель предсказывать метки сущностей.

**Результат:** ELMo значительно улучшает качество NER, потому что контекстуализированные эмбеддинги лучше различают значения слов.

### 5.4 Пример: вопросно-ответные системы (QA)

Для задачи QA (например, SQuAD):

1. Пропускаем вопрос и контекст через biLM.
2. Получаем ELMo-эмбеддинги для вопроса и контекста.
3. Подаём их в модель, которая предсказывает начало и конец ответа в контексте.

**Результат:** в оригинальной статье ELMo улучшил F1 на SQuAD с 81.1% до 85.8% — это значительный прирост.

### 5.5 Результаты ELMo

В оригинальной статье ELMo улучшил state-of-the-art на **шести** NLP-задачах:

| Задача | Метрика | Без ELMo | С ELMo | Прирост |
|--------|---------|----------|--------|---------|
| SQuAD (QA) | F1 | 81.1 | 85.8 | +4.7 |
| SNLI (NLI) | Accuracy | 88.0 | 88.7 | +0.7 |
| SRL (семантические роли) | F1 | 81.7 | 84.6 | +2.9 |
| Coref (кореференция) | F1 | 67.2 | 70.4 | +3.2 |
| NER | F1 | 90.15 | 92.22 | +2.07 |
| SST-5 (тональность) | Accuracy | 51.4 | 54.7 | +3.3 |

**Наблюдение:** ELMo даёт значительный прирост на задачах, где важен контекст: QA, SRL, NER. На задачах, где контекст менее важен (NLI), прирост меньше.

---

## 6. Сравнение ELMo с статическими эмбеддингами

### 6.1 Общие черты

- Все методы дают плотные векторы для слов.
- Все методы обучаются на больших корпусах.
- Все методы можно использовать в downstream-задачах.

### 6.2 Различия

| Свойство | Word2Vec/GloVe/FastText | ELMo |
|----------|------------------------|------|
| Тип | Статические | Контекстуализированные |
| Вектор слова | Один на слово | Зависит от контекста |
| Полисемия | Не решается | Решается |
| OOV | FastText решает | Решает через символы |
| Архитектура | Простая | Глубокая (biLSTM) |
| Обучение | Предсказание контекста | Языковое моделирование |
| Использование | Замена embedding layer | Добавление к модели |
| Вычислительная сложность | Низкая | Высокая |
| Память | Мало | Много |

### 6.3 Пример: полисемия

**Слово «bank»:**

- **Word2Vec:** один вектор, который усредняет значения «финансовый банк» и «речной банк». Ближайшие соседи: «money», «river», «account», «water» — смесь.
- **ELMo:** два разных вектора в зависимости от контекста. В предложении «I went to the bank to deposit money» вектор близок к «money», «account». В предложении «The river bank was flooded» вектор близок к «river», «water».

**Это ключевое преимущество ELMo.**

### 6.4 Пример: OOV

**Слово «unconstitutionality» (не встречалось в корпусе):**

- **Word2Vec/GloVe:** нет вектора.
- **FastText:** вектор строится из символьных n-грамм, но может быть неточным.
- **ELMo:** вектор строится из символов через CNN, что даёт более точное представление.

---

## 7. Ограничения ELMo

### 7.1 Вычислительная сложность

ELMo требует biLSTM с 2 слоями и 4096 единиц. Это означает, что прямой проход по предложению требует значительных вычислений. Для длинных предложений это может быть медленно.

### 7.2 Память

ELMo хранит все скрытые состояния для всех слоёв. Для длинных предложений и больших батчей это требует много памяти.

### 7.3 Не параллелизуется

LSTM обрабатывает последовательность **последовательно**: каждое слово зависит от предыдущего. Это означает, что нельзя распараллелить вычисления по длине последовательности. Это ключевое ограничение, которое было решено в трансформерах (BERT).

### 7.4 Двунаправленность «поверхностная»

ELMo использует два независимых LSTM (прямой и обратный), а затем конкатенирует их. Это не то же самое, что истинно двунаправленная модель, где прямой и обратный контекст взаимодействуют на каждом слое. В BERT это ограничение решено через self-attention.

### 7.5 Дорогое предобучение

Предобучение ELMo на большом корпусе требует много ресурсов (GPU, время). Это ограничивает возможность обучения ELMo на специфических доменах.

---

## 8. Расширения и наследие ELMo

### 8.1 ELMo для других языков

ELMo был обучен не только на английском, но и на других языках: немецком, испанском, португальском и т.д. Для каждого языка требуется свой корпус предобучения.

### 8.2 ELMo для специфических доменов

ELMo можно дообучить на специфическом домене (медицина, юриспруденция, финансы). Это улучшает качество на задачах в этом домене.

### 8.3 Влияние на BERT

ELMo показал, что предобучение языковой модели на большом корпусе и последующий transfer learning — это мощный подход. BERT развил эту идею, заменив LSTM на трансформеры и используя маскированное языковое моделирование.

### 8.4 ELMo в современных моделях

Хотя ELMo был вытеснен BERT и его вариантами, идеи ELMo (контекстуализация, многослойность, transfer learning) живут в современных моделях. Многие современные модели используют комбинацию слоёв, как ELMo.

---

## 9. Практические рекомендации

### 9.1 Когда использовать ELMo

- **Задачи, где важен контекст:** QA, NER, SRL, кореференция.
- **Малые данные:** ELMo можно использовать как feature extractor без дообучения.
- **Специфические домены:** ELMo можно дообучить на домене.

### 9.2 Когда не использовать ELMo

- **Задачи, где контекст не важен:** классификация документов по темам.
- **Ограниченные ресурсы:** ELMo требует много памяти и вычислений.
- **Задачи реального времени:** ELMo медленный.

### 9.3 Как использовать ELMo

1. **Установить AllenNLP:** `pip install allennlp`
2. **Загрузить предобученную модель:** `elmo = ElmoEmbedder()`
3. **Получить эмбеддинги:** `embeddings = elmo.sents2elmo([sentence])`
4. **Использовать в модели:** подать эмбеддинги в downstream-модель.

### 9.4 Гиперпараметры

- **Размерность:** 512 (по умолчанию).
- **Слои:** 2 слоя LSTM (по умолчанию).
- **Веса $s_j^{task}$:** инициализировать равномерно.
- **$\gamma^{task}$:** инициализировать 1.

---

## 10. Заключение

ELMo — это поворотный момент в истории NLP. Он показал, что контекстуализированные эмбеддинги возможны и что они значительно улучшают качество на широком спектре задач. Ключевые идеи ELMo:

1. **Языковое моделирование как задача предобучения.** ELMo обучается предсказывать слова в прямом и обратном направлении.

2. **Символьные свёртки.** ELMo решает проблему OOV через символьные эмбеддинги.

3. **Двунаправленный LSTM.** ELMo использует как левый, так и правый контекст.

4. **Многослойность.** ELMo комбинирует все слои biLM, а не только последний.

5. **Обучаемые веса.** Веса $s_j^{task}$ и $\gamma^{task}$ обучаются для каждой задачи.

**Ключевые формулы:**

Прямое языковое моделирование:

$$
P(t_1, \ldots, t_N) = \prod_{k=1}^{N} P(t_k \mid t_1, \ldots, t_{k-1}).
$$

Обратное языковое моделирование:

$$
P(t_1, \ldots, t_N) = \prod_{k=1}^{N} P(t_k \mid t_{k+1}, \ldots, t_N).
$$

Скрытые состояния biLM:

$$
h_{k,j}^{LM} = [\overrightarrow{h}_{k,j}^{LM}; \overleftarrow{h}_{k,j}^{LM}], \quad j = 1, \ldots, L.
$$

Формула ELMo:

$$
\text{ELMo}_k^{task} = \gamma^{task} \sum_{j=0}^{L} s_j^{task} \mathbf{h}_{k,j}^{LM}.
$$

**Наследие:** ELMo проложил путь для BERT, GPT и всей эпохи предобученных языковых моделей. Его идеи — контекстуализация, многослойность, transfer learning — стали стандартом в современном NLP. Понимание ELMo необходимо для понимания того, как работают современные модели.

---

**В следующей части** мы разберём **BERT** — модель, которая развила идеи ELMo, заменив LSTM на трансформеры и используя маскированное языковое моделирование. Это позволило достичь ещё более высокого качества и параллелизовать вычисления.

# Численный пример Attention: полный пошаговый разбор

## 1. Откуда берутся размерности

Прежде чем начать вычисления, разберёмся, откуда берутся все числа. Это важно, потому что без понимания размерностей пример превращается в магию.

### 1.1 $d_{\text{model}}$ — размерность модели

**Что это:** размерность входных эмбеддингов и всех внутренних представлений в трансформере.

**Откуда берётся:** это гиперпараметр, который выбирает исследователь. В реальных моделях:

- BERT-base: $d_{\text{model}} = 768$;
- GPT-3: $d_{\text{model}} = 12288$;
- Llama 2 7B: $d_{\text{model}} = 4096$.

**Почему именно такие числа:** компромисс между ёмкостью (больше — лучше качество) и вычислительной сложностью (больше — медленнее, больше памяти). Часто выбирают степени двойки для эффективности на GPU.

**В нашем примере:** $d_{\text{model}} = 4$. Это искусственно заниженное значение, чтобы все вычисления можно было проверить вручную.

### 1.2 $h$ — число голов

**Что это:** число параллельных attention-голов в multi-head attention.

**Откуда берётся:** гиперпараметр. В реальных моделях:

- BERT-base: $h = 12$;
- GPT-3: $h = 96$;
- Llama 2 7B: $h = 32$.

**Как выбирают:** обычно так, чтобы $d_k = d_{\text{model}} / h$ было в диапазоне 64–128. Это эмпирическая рекомендация.

**В нашем примере:** $h = 2$.

### 1.3 $d_k$ — размерность запросов и ключей

**Что это:** размерность векторов $Q$ и $K$ после проекции.

**Откуда берётся:** **производная** от $d_{\text{model}}$ и $h$:

$$
d_k = \frac{d_{\text{model}}}{h}.
$$

**Почему так:** чтобы общее число параметров multi-head attention совпадало с числом параметров одного head с размерностью $d_{\text{model}}$. Это архитектурное требование.

**В нашем примере:** $d_k = 4 / 2 = 2$.

### 1.4 $d_v$ — размерность значений

**Что это:** размерность векторов $V$ после проекции.

**Откуда берётся:** обычно $d_v = d_k = d_{\text{model}} / h$.

**Почему так:** чтобы после конкатенации $h$ голов выход имел размерность $h \cdot d_v = d_{\text{model}}$. Это позволяет добавить residual connection.

**В нашем примере:** $d_v = 2$.

### 1.5 $n$ — длина последовательности

**Что это:** число токенов во входной последовательности.

**Откуда берётся:** свойство данных, а не гиперпараметр. Ограничено вычислительными ресурсами (attention квадратичен по $n$).

**В нашем примере:** $n = 3$ (три токена: «кошка», «спит», «на»).

### 1.6 Сводка размерностей

| Размерность | Значение | Откуда |
|-------------|----------|--------|
| $d_{\text{model}}$ | 4 | Гиперпараметр |
| $h$ | 2 | Гиперпараметр |
| $d_k$ | 2 | $d_{\text{model}} / h$ |
| $d_v$ | 2 | $d_{\text{model}} / h$ |
| $n$ | 3 | Длина данных |

---

## 2. Постановка задачи

Рассмотрим предложение из трёх токенов:

$$
\text{«кошка», «спит», «на»}
$$

**Цель:** вычислить attention для этих трёх токенов.

**Параметры:**

- $d_{\text{model}} = 4$;
- $h = 2$;
- $d_k = d_v = 2$;
- $n = 3$.

**Тонкий момент:** мы сначала рассмотрим **single-head attention** (одну голову), а затем **multi-head attention** (две головы). Это позволит увидеть разницу.

---

## 3. Входные эмбеддинги

Пусть после embedding layer мы получили три вектора размерности $d_{\text{model}} = 4$:

$$
x_1 = (1, 0, 0, 1) \quad \text{(кошка)},
$$

$$
x_2 = (0, 1, 1, 0) \quad \text{(спит)},
$$

$$
x_3 = (1, 1, 0, 0) \quad \text{(на)}.
$$

Запишем их в виде матрицы $X \in \mathbb{R}^{3 \times 4}$:

$$
X = \begin{pmatrix}
1 & 0 & 0 & 1 \\
0 & 1 & 1 & 0 \\
1 & 1 & 0 & 0
\end{pmatrix}.
$$

**Тонкий момент:** в реальных моделях к эмбеддингам добавляется positional encoding. Мы пока его опустим, чтобы сосредоточиться на attention.

**Интерпретация:** каждая строка — это вектор одного токена. Компоненты вектора — это «признаки» токена, которые выучил embedding layer.

---

## 4. Single-Head Attention

### 4.1 Матрицы весов

Зададим три матрицы весов для одной головы:

$$
W^Q = \begin{pmatrix}
1 & 0 \\
0 & 1 \\
0 & 0 \\
0 & 0
\end{pmatrix} \in \mathbb{R}^{4 \times 2},
$$

$$
W^K = \begin{pmatrix}
1 & 0 \\
0 & 1 \\
0 & 0 \\
0 & 0
\end{pmatrix} \in \mathbb{R}^{4 \times 2},
$$

$$
W^V = \begin{pmatrix}
1 & 0 \\
0 & 1 \\
0 & 0 \\
0 & 0
\end{pmatrix} \in \mathbb{R}^{4 \times 2}.
$$

**Тонкий момент:** в реальных моделях эти матрицы **обучаются**. Мы задаём их вручную для наглядности. Эти конкретные матрицы «выбирают» первые две компоненты входа.

**Почему такие:** это простейший случай, при котором $Q, K, V$ — это просто первые две компоненты $X$. Это упрощает вычисления.

### 4.2 Вычисление Q

$$
Q = X W^Q \in \mathbb{R}^{3 \times 2}.
$$

**Строка 1 (кошка):**

$$
q_1 = x_1 W^Q = (1, 0, 0, 1) \begin{pmatrix} 1 & 0 \\ 0 & 1 \\ 0 & 0 \\ 0 & 0 \end{pmatrix}.
$$

Первая компонента:

$$
1 \cdot 1 + 0 \cdot 0 + 0 \cdot 0 + 1 \cdot 0 = 1.
$$

Вторая компонента:

$$
1 \cdot 0 + 0 \cdot 1 + 0 \cdot 0 + 1 \cdot 0 = 0.
$$

$$
q_1 = (1, 0).
$$

**Строка 2 (спит):**

$$
q_2 = x_2 W^Q = (0, 1, 1, 0) \begin{pmatrix} 1 & 0 \\ 0 & 1 \\ 0 & 0 \\ 0 & 0 \end{pmatrix}.
$$

Первая компонента:

$$
0 \cdot 1 + 1 \cdot 0 + 1 \cdot 0 + 0 \cdot 0 = 0.
$$

Вторая компонента:

$$
0 \cdot 0 + 1 \cdot 1 + 1 \cdot 0 + 0 \cdot 0 = 1.
$$

$$
q_2 = (0, 1).
$$

**Строка 3 (на):**

$$
q_3 = x_3 W^Q = (1, 1, 0, 0) \begin{pmatrix} 1 & 0 \\ 0 & 1 \\ 0 & 0 \\ 0 & 0 \end{pmatrix}.
$$

Первая компонента:

$$
1 \cdot 1 + 1 \cdot 0 + 0 \cdot 0 + 0 \cdot 0 = 1.
$$

Вторая компонента:

$$
1 \cdot 0 + 1 \cdot 1 + 0 \cdot 0 + 0 \cdot 0 = 1.
$$

$$
q_3 = (1, 1).
$$

**Итог:**

$$
Q = \begin{pmatrix} 1 & 0 \\ 0 & 1 \\ 1 & 1 \end{pmatrix}.
$$

### 4.3 Вычисление K

$$
K = X W^K \in \mathbb{R}^{3 \times 2}.
$$

Поскольку $W^K = W^Q$, получаем то же самое:

$$
K = \begin{pmatrix} 1 & 0 \\ 0 & 1 \\ 1 & 1 \end{pmatrix}.
$$

### 4.4 Вычисление V

$$
V = X W^V \in \mathbb{R}^{3 \times 2}.
$$

Поскольку $W^V = W^Q$, получаем:

$$
V = \begin{pmatrix} 1 & 0 \\ 0 & 1 \\ 1 & 1 \end{pmatrix}.
$$

**Тонкий момент:** в реальных моделях $W^Q, W^K, W^V$ **разные**, поэтому $Q, K, V$ различаются. У нас они совпали из-за выбора матриц.

### 4.5 Вычисление матрицы сходств $Q K^\top$

$$
Q K^\top \in \mathbb{R}^{3 \times 3}.
$$

**Элемент (1,1):** $q_1 \cdot k_1$

$$
q_1 \cdot k_1 = 1 \cdot 1 + 0 \cdot 0 = 1.
$$

**Элемент (1,2):** $q_1 \cdot k_2$

$$
q_1 \cdot k_2 = 1 \cdot 0 + 0 \cdot 1 = 0.
$$

**Элемент (1,3):** $q_1 \cdot k_3$

$$
q_1 \cdot k_3 = 1 \cdot 1 + 0 \cdot 1 = 1.
$$

**Элемент (2,1):** $q_2 \cdot k_1$

$$
q_2 \cdot k_1 = 0 \cdot 1 + 1 \cdot 0 = 0.
$$

**Элемент (2,2):** $q_2 \cdot k_2$

$$
q_2 \cdot k_2 = 0 \cdot 0 + 1 \cdot 1 = 1.
$$

**Элемент (2,3):** $q_2 \cdot k_3$

$$
q_2 \cdot k_3 = 0 \cdot 1 + 1 \cdot 1 = 1.
$$

**Элемент (3,1):** $q_3 \cdot k_1$

$$
q_3 \cdot k_1 = 1 \cdot 1 + 1 \cdot 0 = 1.
$$

**Элемент (3,2):** $q_3 \cdot k_2$

$$
q_3 \cdot k_2 = 1 \cdot 0 + 1 \cdot 1 = 1.
$$

**Элемент (3,3):** $q_3 \cdot k_3$

$$
q_3 \cdot k_3 = 1 \cdot 1 + 1 \cdot 1 = 2.
$$

**Матрица сходств:**

$$
Q K^\top = \begin{pmatrix}
1 & 0 & 1 \\
0 & 1 & 1 \\
1 & 1 & 2
\end{pmatrix}.
$$

**Наблюдение:** матрица **симметрична**, потому что $Q = K$. В реальных моделях симметрии не будет.

### 4.6 Масштабирование

Делим на $\sqrt{d_k} = \sqrt{2} \approx 1.4142$:

$$
\frac{Q K^\top}{\sqrt{2}} = \begin{pmatrix}
0.7071 & 0 & 0.7071 \\
0 & 0.7071 & 0.7071 \\
0.7071 & 0.7071 & 1.4142
\end{pmatrix}.
$$

**Проверка:** $1 / 1.4142 = 0.7071$, $2 / 1.4142 = 1.4142$. Верно.

**Зачем масштабировать:** без масштабирования значения $q \cdot k$ могут быть большими, softmax становится «жёстким», градиенты затухают. Масштабирование на $\sqrt{d_k}$ стабилизирует обучение.

### 4.7 Softmax по строкам

**Строка 1:** значения $(0.7071, 0, 0.7071)$.

$$
\exp(0.7071) = 2.0281,
$$
$$
\exp(0) = 1,
$$
$$
\exp(0.7071) = 2.0281.
$$

Сумма:

$$
Z_1 = 2.0281 + 1 + 2.0281 = 5.0562.
$$

Веса:

$$
\alpha_{11} = \frac{2.0281}{5.0562} = 0.4011,
$$
$$
\alpha_{12} = \frac{1}{5.0562} = 0.1978,
$$
$$
\alpha_{13} = \frac{2.0281}{5.0562} = 0.4011.
$$

**Строка 2:** значения $(0, 0.7071, 0.7071)$.

$$
\exp(0) = 1,
$$
$$
\exp(0.7071) = 2.0281,
$$
$$
\exp(0.7071) = 2.0281.
$$

Сумма:

$$
Z_2 = 1 + 2.0281 + 2.0281 = 5.0562.
$$

Веса:

$$
\alpha_{21} = \frac{1}{5.0562} = 0.1978,
$$
$$
\alpha_{22} = \frac{2.0281}{5.0562} = 0.4011,
$$
$$
\alpha_{23} = \frac{2.0281}{5.0562} = 0.4011.
$$

**Строка 3:** значения $(0.7071, 0.7071, 1.4142)$.

$$
\exp(0.7071) = 2.0281,
$$
$$
\exp(0.7071) = 2.0281,
$$
$$
\exp(1.4142) = 4.1133.
$$

Сумма:

$$
Z_3 = 2.0281 + 2.0281 + 4.1133 = 8.1695.
$$

Веса:

$$
\alpha_{31} = \frac{2.0281}{8.1695} = 0.2482,
$$
$$
\alpha_{32} = \frac{2.0281}{8.1695} = 0.2482,
$$
$$
\alpha_{33} = \frac{4.1133}{8.1695} = 0.5035.
$$

**Матрица attention-весов:**

$$
\alpha = \begin{pmatrix}
0.4011 & 0.1978 & 0.4011 \\
0.1978 & 0.4011 & 0.4011 \\
0.2482 & 0.2482 & 0.5035
\end{pmatrix}.
$$

**Проверка:** суммы по строкам равны 1.

- Строка 1: $0.4011 + 0.1978 + 0.4011 = 1.0000$;
- Строка 2: $0.1978 + 0.4011 + 0.4011 = 1.0000$;
- Строка 3: $0.2482 + 0.2482 + 0.5035 = 1.0000$.

**Интерпретация:**

- Для токена «кошка» (строка 1): веса $0.4011$, $0.1978$, $0.4011$. Модель обращает внимание на «кошку» и «на» одинаково, а на «спит» — меньше.
- Для токена «спит» (строка 2): веса $0.1978$, $0.4011$, $0.4011$. Модель обращает внимание на «спит» и «на» одинаково, а на «кошку» — меньше.
- Для токена «на» (строка 3): веса $0.2482$, $0.2482$, $0.5035$. Модель обращает внимание в основном на себя.

### 4.8 Взвешенная сумма значений

$$
\text{output} = \alpha V \in \mathbb{R}^{3 \times 2}.
$$

**Строка 1 (кошка):**

$$
\text{out}_1 = 0.4011 \cdot v_1 + 0.1978 \cdot v_2 + 0.4011 \cdot v_3.
$$

Первая компонента:

$$
0.4011 \cdot 1 + 0.1978 \cdot 0 + 0.4011 \cdot 1 = 0.4011 + 0 + 0.4011 = 0.8022.
$$

Вторая компонента:

$$
0.4011 \cdot 0 + 0.1978 \cdot 1 + 0.4011 \cdot 1 = 0 + 0.1978 + 0.4011 = 0.5989.
$$

$$
\text{out}_1 = (0.8022, 0.5989).
$$

**Строка 2 (спит):**

$$
\text{out}_2 = 0.1978 \cdot v_1 + 0.4011 \cdot v_2 + 0.4011 \cdot v_3.
$$

Первая компонента:

$$
0.1978 \cdot 1 + 0.4011 \cdot 0 + 0.4011 \cdot 1 = 0.1978 + 0 + 0.4011 = 0.5989.
$$

Вторая компонента:

$$
0.1978 \cdot 0 + 0.4011 \cdot 1 + 0.4011 \cdot 1 = 0 + 0.4011 + 0.4011 = 0.8022.
$$

$$
\text{out}_2 = (0.5989, 0.8022).
$$

**Строка 3 (на):**

$$
\text{out}_3 = 0.2482 \cdot v_1 + 0.2482 \cdot v_2 + 0.5035 \cdot v_3.
$$

Первая компонента:

$$
0.2482 \cdot 1 + 0.2482 \cdot 0 + 0.5035 \cdot 1 = 0.2482 + 0 + 0.5035 = 0.7517.
$$

Вторая компонента:

$$
0.2482 \cdot 0 + 0.2482 \cdot 1 + 0.5035 \cdot 1 = 0 + 0.2482 + 0.5035 = 0.7517.
$$

$$
\text{out}_3 = (0.7517, 0.7517).
$$

**Итоговый output single-head:**

$$
\text{Output}_{\text{single}} = \begin{pmatrix}
0.8022 & 0.5989 \\
0.5989 & 0.8022 \\
0.7517 & 0.7517
\end{pmatrix}.
$$

**Интерпретация:**

- Для «кошки» output $(0.8022, 0.5989)$ — смесь значений, где доминируют «кошка» и «на».
- Для «спита» output $(0.5989, 0.8022)$ — смесь, где доминируют «спит» и «на».
- Для «на» output $(0.7517, 0.7517)$ — смесь, где доминирует само «на».

---

## 5. Multi-Head Attention

### 5.1 Постановка

Теперь рассмотрим **multi-head attention** с $h = 2$ головами. Каждая голова имеет свои проекции $W_i^Q, W_i^K, W_i^V$.

**Размерности:**

- $d_{\text{model}} = 4$;
- $h = 2$;
- $d_k = d_v = 2$.

### 5.2 Голова 1: проекции

**Голова 1** использует **первые две** компоненты входа:

$$
W_1^Q = \begin{pmatrix}
1 & 0 \\
0 & 1 \\
0 & 0 \\
0 & 0
\end{pmatrix}, \quad
W_1^K = \begin{pmatrix}
1 & 0 \\
0 & 1 \\
0 & 0 \\
0 & 0
\end{pmatrix}, \quad
W_1^V = \begin{pmatrix}
1 & 0 \\
0 & 1 \\
0 & 0 \\
0 & 0
\end{pmatrix}.
$$

**Вычисление $Q_1, K_1, V_1$:**

$$
Q_1 = X W_1^Q = \begin{pmatrix} 1 & 0 \\ 0 & 1 \\ 1 & 1 \end{pmatrix},
$$

$$
K_1 = \begin{pmatrix} 1 & 0 \\ 0 & 1 \\ 1 & 1 \end{pmatrix}, \quad
V_1 = \begin{pmatrix} 1 & 0 \\ 0 & 1 \\ 1 & 1 \end{pmatrix}.
$$

### 5.3 Голова 1: attention

Это **тот же** attention, что мы вычислили в single-head. Результат:

$$
\text{Output}_1 = \begin{pmatrix}
0.8022 & 0.5989 \\
0.5989 & 0.8022 \\
0.7517 & 0.7517
\end{pmatrix}.
$$

### 5.4 Голова 2: проекции

**Голова 2** использует **последние две** компоненты входа:

$$
W_2^Q = \begin{pmatrix}
0 & 0 \\
0 & 0 \\
1 & 0 \\
0 & 1
\end{pmatrix}, \quad
W_2^K = \begin{pmatrix}
0 & 0 \\
0 & 0 \\
1 & 0 \\
0 & 1
\end{pmatrix}, \quad
W_2^V = \begin{pmatrix}
0 & 0 \\
0 & 0 \\
1 & 0 \\
0 & 1
\end{pmatrix}.
$$

**Вычисление $Q_2$:**

**Строка 1 (кошка):**

$$
q_1 = x_1 W_2^Q = (1, 0, 0, 1) \begin{pmatrix} 0 & 0 \\ 0 & 0 \\ 1 & 0 \\ 0 & 1 \end{pmatrix}.
$$

Первая компонента:

$$
1 \cdot 0 + 0 \cdot 0 + 0 \cdot 1 + 1 \cdot 0 = 0.
$$

Вторая компонента:

$$
1 \cdot 0 + 0 \cdot 0 + 0 \cdot 0 + 1 \cdot 1 = 1.
$$

$$
q_1 = (0, 1).
$$

**Строка 2 (спит):**

$$
q_2 = (0, 1, 1, 0) \begin{pmatrix} 0 & 0 \\ 0 & 0 \\ 1 & 0 \\ 0 & 1 \end{pmatrix}.
$$

Первая компонента:

$$
0 \cdot 0 + 1 \cdot 0 + 1 \cdot 1 + 0 \cdot 0 = 1.
$$

Вторая компонента:

$$
0 \cdot 0 + 1 \cdot 0 + 1 \cdot 0 + 0 \cdot 1 = 0.
$$

$$
q_2 = (1, 0).
$$

**Строка 3 (на):**

$$
q_3 = (1, 1, 0, 0) \begin{pmatrix} 0 & 0 \\ 0 & 0 \\ 1 & 0 \\ 0 & 1 \end{pmatrix}.
$$

Первая компонента:

$$
1 \cdot 0 + 1 \cdot 0 + 0 \cdot 1 + 0 \cdot 0 = 0.
$$

Вторая компонента:

$$
1 \cdot 0 + 1 \cdot 0 + 0 \cdot 0 + 0 \cdot 1 = 0.
$$

$$
q_3 = (0, 0).
$$

**Итог $Q_2$:**

$$
Q_2 = \begin{pmatrix} 0 & 1 \\ 1 & 0 \\ 0 & 0 \end{pmatrix}.
$$

**Аналогично $K_2 = V_2 = Q_2$:**

$$
K_2 = \begin{pmatrix} 0 & 1 \\ 1 & 0 \\ 0 & 0 \end{pmatrix}, \quad
V_2 = \begin{pmatrix} 0 & 1 \\ 1 & 0 \\ 0 & 0 \end{pmatrix}.
$$

### 5.5 Голова 2: attention

**Матрица сходств $Q_2 K_2^\top$:**

**Элемент (1,1):** $q_1 \cdot k_1 = 0 \cdot 0 + 1 \cdot 1 = 1$.

**Элемент (1,2):** $q_1 \cdot k_2 = 0 \cdot 1 + 1 \cdot 0 = 0$.

**Элемент (1,3):** $q_1 \cdot k_3 = 0 \cdot 0 + 1 \cdot 0 = 0$.

**Элемент (2,1):** $q_2 \cdot k_1 = 1 \cdot 0 + 0 \cdot 1 = 0$.

**Элемент (2,2):** $q_2 \cdot k_2 = 1 \cdot 1 + 0 \cdot 0 = 1$.

**Элемент (2,3):** $q_2 \cdot k_3 = 1 \cdot 0 + 0 \cdot 0 = 0$.

**Элемент (3,1):** $q_3 \cdot k_1 = 0 \cdot 0 + 0 \cdot 1 = 0$.

**Элемент (3,2):** $q_3 \cdot k_2 = 0 \cdot 1 + 0 \cdot 0 = 0$.

**Элемент (3,3):** $q_3 \cdot k_3 = 0 \cdot 0 + 0 \cdot 0 = 0$.

$$
Q_2 K_2^\top = \begin{pmatrix}
1 & 0 & 0 \\
0 & 1 & 0 \\
0 & 0 & 0
\end{pmatrix}.
$$

**Масштабирование:**

$$
\frac{Q_2 K_2^\top}{\sqrt{2}} = \begin{pmatrix}
0.7071 & 0 & 0 \\
0 & 0.7071 & 0 \\
0 & 0 & 0
\end{pmatrix}.
$$

**Softmax по строкам:**

**Строка 1:** $\exp(0.7071) = 2.0281$, $\exp(0) = 1$, $\exp(0) = 1$. Сумма $= 4.0281$.

$$
\alpha_1 = (0.5035, 0.2482, 0.2482).
$$

**Строка 2:** аналогично.

$$
\alpha_2 = (0.2482, 0.5035, 0.2482).
$$

**Строка 3:** все нули, softmax даёт равномерное распределение.

$$
\alpha_3 = (0.3333, 0.3333, 0.3333).
$$

**Матрица attention-весов головы 2:**

$$
\alpha^{(2)} = \begin{pmatrix}
0.5035 & 0.2482 & 0.2482 \\
0.2482 & 0.5035 & 0.2482 \\
0.3333 & 0.3333 & 0.3333
\end{pmatrix}.
$$

**Взвешенная сумма значений:**

**Строка 1:**

$$
0.5035 \cdot (0, 1) + 0.2482 \cdot (1, 0) + 0.2482 \cdot (0, 0) = (0.2482, 0.5035).
$$

**Строка 2:**

$$
0.2482 \cdot (0, 1) + 0.5035 \cdot (1, 0) + 0.2482 \cdot (0, 0) = (0.5035, 0.2482).
$$

**Строка 3:**

$$
0.3333 \cdot (0, 1) + 0.3333 \cdot (1, 0) + 0.3333 \cdot (0, 0) = (0.3333, 0.3333).
$$

$$
\text{Output}_2 = \begin{pmatrix}
0.2482 & 0.5035 \\
0.5035 & 0.2482 \\
0.3333 & 0.3333
\end{pmatrix}.
$$

### 5.6 Конкатенация голов

$$
\text{Concat} = [\text{Output}_1; \text{Output}_2] \in \mathbb{R}^{3 \times 4}.
$$

**Строка 1:**

$$
[0.8022, 0.5989, 0.2482, 0.5035].
$$

**Строка 2:**

$$
[0.5989, 0.8022, 0.5035, 0.2482].
$$

**Строка 3:**

$$
[0.7517, 0.7517, 0.3333, 0.3333].
$$

$$
\text{Concat} = \begin{pmatrix}
0.8022 & 0.5989 & 0.2482 & 0.5035 \\
0.5989 & 0.8022 & 0.5035 & 0.2482 \\
0.7517 & 0.7517 & 0.3333 & 0.3333
\end{pmatrix}.
$$

### 5.7 Проекция выхода

Для простоты используем $W^O = I_4$ (единичная матрица). Тогда:

$$
\text{MultiHeadOutput} = \text{Concat}.
$$

**Тонкий момент:** в реальных моделях $W^O$ — обучаемая матрица, которая смешивает информацию от разных голов.

**Интерпретация:** каждая голова улавливает разные аспекты. Голова 1 работает с первыми двумя компонентами, голова 2 — с последними двумя. В реальных моделях головы **не разделены** так явно: они используют разные проекции одних и тех же входов, и их специализация возникает в процессе обучения.

---

## 6. Маскирование

### 6.1 Causal mask

Применим causal mask к scaled scores головы 1:

$$
M = \begin{pmatrix}
0 & -\infty & -\infty \\
0 & 0 & -\infty \\
0 & 0 & 0
\end{pmatrix}.
$$

Складываем с scaled scores:

$$
\text{scores} + M = \begin{pmatrix}
0.7071 & -\infty & -\infty \\
0 & 0.7071 & -\infty \\
0.7071 & 0.7071 & 1.4142
\end{pmatrix}.
$$

**Softmax:**

**Строка 1:** $\exp(0.7071) = 2.0281$, $\exp(-\infty) = 0$, $\exp(-\infty) = 0$. Сумма $= 2.0281$.

$$
\alpha_1 = (1.0000, 0, 0).
$$

**Строка 2:** $\exp(0) = 1$, $\exp(0.7071) = 2.0281$, $\exp(-\infty) = 0$. Сумма $= 3.0281$.

$$
\alpha_2 = (0.3302, 0.6698, 0).
$$

**Строка 3:** $\exp(0.7071) = 2.0281$, $\exp(0.7071) = 2.0281$, $\exp(1.4142) = 4.1133$. Сумма $= 8.1695$.

$$
\alpha_3 = (0.2482, 0.2482, 0.5035).
$$

**Output с causal mask:**

**Строка 1:**

$$
1.0000 \cdot (1, 0) + 0 \cdot (0, 1) + 0 \cdot (1, 1) = (1, 0).
$$

**Строка 2:**

$$
0.3302 \cdot (1, 0) + 0.6698 \cdot (0, 1) + 0 \cdot (1, 1) = (0.3302, 0.6698).
$$

**Строка 3:**

$$
0.2482 \cdot (1, 0) + 0.2482 \cdot (0, 1) + 0.5035 \cdot (1, 1) = (0.7517, 0.7517).
$$

**Интерпретация:** токен «кошка» (строка 1) теперь обращает внимание **только** на себя. Токен «спит» (строка 2) — на «кошку» и себя. Токен «на» (строка 3) — на все три токена.

### 6.2 Padding mask

Предположим, что третий токен — это `<PAD>`. Применим padding mask, запрещающий внимание к третьему столбцу:

$$
M = \begin{pmatrix}
0 & 0 & -\infty \\
0 & 0 & -\infty \\
0 & 0 & -\infty
\end{pmatrix}.
$$

**Softmax:**

**Строка 1:** $\exp(0.7071) = 2.0281$, $\exp(0) = 1$, $\exp(-\infty) = 0$. Сумма $= 3.0281$.

$$
\alpha_1 = (0.6698, 0.3302, 0).
$$

**Строка 2:** $\exp(0) = 1$, $\exp(0.7071) = 2.0281$, $\exp(-\infty) = 0$. Сумма $= 3.0281$.

$$
\alpha_2 = (0.3302, 0.6698, 0).
$$

**Строка 3:** $\exp(0.7071) = 2.0281$, $\exp(0.7071) = 2.0281$, $\exp(-\infty) = 0$. Сумма $= 4.0562$.

$$
\alpha_3 = (0.5, 0.5, 0).
$$

**Output с padding mask:**

**Строка 1:**

$$
0.6698 \cdot (1, 0) + 0.3302 \cdot (0, 1) + 0 \cdot (1, 1) = (0.6698, 0.3302).
$$

**Строка 2:**

$$
0.3302 \cdot (1, 0) + 0.6698 \cdot (0, 1) + 0 \cdot (1, 1) = (0.3302, 0.6698).
$$

**Строка 3:**

$$
0.5 \cdot (1, 0) + 0.5 \cdot (0, 1) + 0 \cdot (1, 1) = (0.5, 0.5).
$$

**Интерпретация:** `<PAD>`-токен (третий) больше не влияет на output. Веса для него равны нулю.

---

## 7. Проверка размерностей

### 7.1 Single-head

| Этап | Размерность |
|------|-------------|
| Вход $X$ | $3 \times 4$ |
| $W^Q, W^K, W^V$ | $4 \times 2$ |
| $Q, K, V$ | $3 \times 2$ |
| $Q K^\top$ | $3 \times 3$ |
| $\alpha$ | $3 \times 3$ |
| Output | $3 \times 2$ |

### 7.2 Multi-head

| Этап | Размерность |
|------|-------------|
| Вход $X$ | $3 \times 4$ |
| $W_i^Q, W_i^K, W_i^V$ | $4 \times 2$ |
| $Q_i, K_i, V_i$ | $3 \times 2$ |
| $Q_i K_i^\top$ | $3 \times 3$ |
| $\text{head}_i$ | $3 \times 2$ |
| Concat | $3 \times 4$ |
| $W^O$ | $4 \times 4$ |
| Output | $3 \times 4$ |

---

## 8. Полная сводка вычислений

### 8.1 Single-head

| Этап | Токен «кошка» | Токен «спит» | Токен «на» |
|------|---------------|--------------|------------|
| Вход $x$ | $(1, 0, 0, 1)$ | $(0, 1, 1, 0)$ | $(1, 1, 0, 0)$ |
| Query $q$ | $(1, 0)$ | $(0, 1)$ | $(1, 1)$ |
| Key $k$ | $(1, 0)$ | $(0, 1)$ | $(1, 1)$ |
| Value $v$ | $(1, 0)$ | $(0, 1)$ | $(1, 1)$ |
| $q \cdot k_1$ | 1 | 0 | 1 |
| $q \cdot k_2$ | 0 | 1 | 1 |
| $q \cdot k_3$ | 1 | 1 | 2 |
| Softmax | $(0.4011, 0.1978, 0.4011)$ | $(0.1978, 0.4011, 0.4011)$ | $(0.2482, 0.2482, 0.5035)$ |
| Output | $(0.8022, 0.5989)$ | $(0.5989, 0.8022)$ | $(0.7517, 0.7517)$ |

### 8.2 Multi-head

| Этап | Токен «кошка» | Токен «спит» | Токен «на» |
|------|---------------|--------------|------------|
| Вход $x$ | $(1, 0, 0, 1)$ | $(0, 1, 1, 0)$ | $(1, 1, 0, 0)$ |
| Head 1 output | $(0.8022, 0.5989)$ | $(0.5989, 0.8022)$ | $(0.7517, 0.7517)$ |
| Head 2 output | $(0.2482, 0.5035)$ | $(0.5035, 0.2482)$ | $(0.3333, 0.3333)$ |
| Concat | $(0.8022, 0.5989, 0.2482, 0.5035)$ | $(0.5989, 0.8022, 0.5035, 0.2482)$ | $(0.7517, 0.7517, 0.3333, 0.3333)$ |

---

## 9. Интерпретация результатов

### 9.1 Single-head

**Для токена «кошка»:** attention-веса $(0.4011, 0.1978, 0.4011)$ показывают, что модель обращает внимание на «кошку» и «на» одинаково, а на «спит» — меньше. Output $(0.8022, 0.5989)$ — это смесь значений.

**Для токена «спит»:** attention-веса $(0.1978, 0.4011, 0.4011)$ показывают, что модель обращает внимание на «спит» и «на», а на «кошку» — меньше.

**Для токена «на»:** attention-веса $(0.2482, 0.2482, 0.5035)$ показывают, что модель обращает внимание в основном на себя.

### 9.2 Multi-head

**Голова 1** работает с первыми двумя компонентами входа. Она улавливает связи между «кошкой», «спитом» и «на».

**Голова 2** работает с последними двумя компонентами. Она улавливает другие связи: «кошка» и «спит» обращают внимание на себя, «на» — равномерно на всех.

**Конкатенация** даёт вектор размерности 4, который содержит информацию от обеих голов. В реальных моделях $W^O$ дополнительно смешивает эту информацию.

### 9.3 Почему в нашем примере веса близки к равномерным

В реальных моделях attention-веса часто более контрастны: один токен получает вес 0.8–0.9, остальные — 0.05–0.1. У нас веса близки к равномерным, потому что:

1. Входные векторы заданы вручную и не имеют ярко выраженной структуры.
2. Матрицы весов заданы вручную и не обучены.
3. Размерности маленькие, поэтому различия сглажены.

В реальных моделях после обучения веса становятся более контрастными, потому что модель учится выделять важные связи.

---

## 10. Заключение

В этом численном примере мы шаг за шагом вычислили attention для предложения из трёх токенов. Основные выводы:

1. **Query, Key, Value** — это проекции входных эмбеддингов через обучаемые матрицы $W^Q, W^K, W^V$.

2. **Scaled dot-product attention** вычисляет сходство между запросами и ключами, нормализует через softmax и возвращает взвешенную сумму значений.

3. **Softmax** превращает сходства в распределение вероятностей. Сумма весов по строке равна 1.

4. **Attention-веса** показывают, на какие токены модель обращает внимание.

5. **Multi-head attention** позволяет модели смотреть на разные аспекты. Каждая голова использует свои проекции.

6. **Маскирование** позволяет исключить нежелательные позиции: causal mask для авторегрессии, padding mask для батчей разной длины.

7. **Размерности** — это гиперпараметры. $d_{\text{model}}$ выбирается как компромисс между ёмкостью и скоростью. $h$ выбирается так, чтобы $d_k = d_{\text{model}} / h \approx 64$–$128$. $d_k = d_v = d_{\text{model}} / h$ — производные.

**Ключевые формулы:**

Scaled dot-product attention:
$$
\text{Attention}(Q, K, V) = \text{softmax}\left( \frac{Q K^\top}{\sqrt{d_k}} \right) V.
$$

Multi-head attention:
$$
\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, \ldots, \text{head}_h) W^O,
$$
$$
\text{head}_i = \text{Attention}(Q W_i^Q, K W_i^K, V W_i^V).
$$

Маскирование:
$$
\text{Attention}(Q, K, V) = \text{softmax}\left( \frac{Q K^\top}{\sqrt{d_k}} + M \right) V.
$$

Этот пример, несмотря на маленькие размерности, показывает все ключевые шаги attention. В реальных моделях размерности больше ($d_{\text{model}} = 512$, $h = 8$, $d_k = 64$), но математика остаётся той же.

---

**В следующей части** мы разберём **Transformer** — архитектуру, построенную на attention. Мы рассмотрим encoder, decoder, positional encoding, layer normalization и residual connections.